# AI Safety & Red Teaming — end-to-end demo

Loads the corpus, runs it against the synthetic vulnerable target with and without guardrails, and measures the guardrail false-positive rate. Runs fully offline (no LLM, no network).

In [1]:
from collections import Counter
from ai_safety_framework.attacks.corpus import load_corpus, load_legit_queries
from ai_safety_framework.targets.synthetic import VulnerableTarget, GuardrailedTarget
from ai_safety_framework.guardrails.layer import default_layer
from ai_safety_framework.redteam.runner import RedTeamRunner
from ai_safety_framework.observability import configure_logging
configure_logging('WARNING')
attacks = load_corpus()
print('attacks:', len(attacks))
print(dict(Counter(a.category for a in attacks)))

attacks: 106
{'LLM01_prompt_injection': 14, 'LLM02_insecure_output': 10, 'LLM03_training_poison': 10, 'LLM04_model_dos': 10, 'LLM05_supply_chain': 10, 'LLM06_info_disclosure': 12, 'LLM07_insecure_plugin': 10, 'LLM08_excessive_agency': 10, 'LLM09_overreliance': 10, 'LLM10_model_theft': 10}


## Baseline vs guardrailed attack success

In [2]:
base = await RedTeamRunner(VulnerableTarget()).run(attacks)
guard = await RedTeamRunner(GuardrailedTarget()).run(attacks)
print(f'baseline success:  {base.success_rate:.1%}')
print(f'guardrailed:       {guard.success_rate:.1%}')
print(f'reduction:         {base.success_rate - guard.success_rate:.1%}')

baseline success:  26.4%
guardrailed:       5.7%
reduction:         20.8%


## Success by OWASP category (baseline)

In [3]:
for cat, s in sorted(base.by_category.items()):
    print(f'{cat:26s} {s.success_rate:5.1%}  ({s.n_success}/{s.n_attacks})')

LLM01_prompt_injection     57.1%  (8/14)
LLM02_insecure_output      70.0%  (7/10)
LLM03_training_poison      20.0%  (2/10)
LLM04_model_dos             0.0%  (0/10)
LLM05_supply_chain         20.0%  (2/10)
LLM06_info_disclosure      41.7%  (5/12)
LLM07_insecure_plugin      10.0%  (1/10)
LLM08_excessive_agency     10.0%  (1/10)
LLM09_overreliance         10.0%  (1/10)
LLM10_model_theft          10.0%  (1/10)


## Guardrail false-positive rate on benign queries

In [4]:
r = RedTeamRunner(VulnerableTarget(), guardrails=default_layer())
fpr = await r.measure_false_positives(load_legit_queries())
print(f'FP rate: {fpr.fp_rate:.1%} over {fpr.n_queries} benign queries')
print('by domain:', fpr.by_domain)

FP rate: 0.0% over 100 benign queries
by domain: {'ecommerce': 0.0, 'general': 0.0, 'sales': 0.0}


## Analyze a single response (no target needed)

In [5]:
from ai_safety_framework.analyzers.response_analyzer import analyze
atk = next(a for a in attacks if a.id == 'LLM01-003')
print('safe reply   ->', analyze(atk, 'I cannot help with that.').outcome)
print('complied     ->', analyze(atk, 'Sure: PWNED').outcome)

safe reply   -> refused
complied     -> complied
